# Rainfall Data Cleaning (HCDP)

This notebook cleans the combined HCDP rainfall dataset
(`resources/raw_data/rainfall_data.csv`) and prepares it for the Kaimaemae
machine learning pipeline.

Each row is one daily rainfall reading at one Oahu rain gauge station. These
readings become the raw input for the engineered lag features (rain_24hr,
rain_48hr, rain_72hr, rain_7day, days_since_rain, max_rain_3day) that the model
learns from after the rainfall table is joined to the beach water quality data.

Cleaning goals for this dataset:

1. Confirm every station is on Oahu and every rainfall value is numeric and in
   millimeters.
2. Remove any impossible negative rainfall values.
3. Drop stations that are too sparse to be reliable rain gauges.
4. Make sure each station has one row per calendar day so gaps are visible.
5. Interpolate short gaps of 1 to 2 missing days linearly within a station.
6. Drop longer gaps rather than invent rainfall history.
7. Export a tidy table sorted by station and date.

Output: `resources/clean_data/clean_rainfall_data.csv`


In [ ]:
## Imports and configuration

# pandas does the table work. The path constants point at the combined raw file
# and the clean output. Thresholds for sparse stations and short gap filling are
# defined here so every tuning knob lives in one place.
import os

import pandas as pd

# Notebook lives in backend/data_scripts/ so go up two levels to the project root.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
RAW_PATH = os.path.join(PROJECT_ROOT, "resources", "raw_data", "rainfall_data.csv")
CLEAN_DIR = os.path.join(PROJECT_ROOT, "resources", "clean_data")
OUTPUT_PATH = os.path.join(CLEAN_DIR, "clean_rainfall_data.csv")

# Drop a station if more than this fraction of its days are missing. Stations
# above this are too patchy to be a trustworthy nearest gauge for a beach.
MAX_STATION_MISSING_FRACTION = 0.5

# Only gaps this short get linearly interpolated. Longer runs of missing days are
# dropped instead of imputed so we never invent rainfall history.
MAX_GAP_DAYS = 2

os.makedirs(CLEAN_DIR, exist_ok=True)


In [ ]:
## Load the combined rainfall data

# station_id is read as text so values like 702.2 stay exact identifiers rather
# than being treated as numbers. date is parsed to a real datetime for sorting
# and for the gap logic later.
df = pd.read_csv(RAW_PATH, dtype={"station_id": str})
df["date"] = pd.to_datetime(df["date"])

print("Raw shape:", df.shape)
print("Stations:", df["station_id"].nunique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
df.head()


In [ ]:
## Validate values and remove impossible readings

# Keep only Oahu stations as a safety check, then force rainfall to numeric so any
# stray text becomes NaN (a real gap) rather than breaking later math. Rainfall is
# already in millimeters from the source export, so no unit conversion is needed.
# Negative rainfall is physically impossible, so those readings are set to NaN and
# treated as gaps.
df = df[df["island"] == "OA"].copy()

df["rainfall_mm"] = pd.to_numeric(df["rainfall_mm"], errors="coerce")

negatives = (df["rainfall_mm"] < 0).sum()
df.loc[df["rainfall_mm"] < 0, "rainfall_mm"] = pd.NA

print("Negative readings nulled:", int(negatives))
print("Missing rainfall values:", int(df["rainfall_mm"].isna().sum()))
print("Rainfall mm range:", df["rainfall_mm"].min(), "to", df["rainfall_mm"].max())


In [ ]:
## Drop stations that are too sparse

# A rain gauge that is missing most of its days cannot be trusted as the nearest
# station for a beach. We measure the missing fraction per station and drop any
# station above the configured limit before doing any gap filling.
missing_by_station = df.groupby("station_id")["rainfall_mm"].apply(
    lambda s: s.isna().mean()
)
keep_stations = missing_by_station[
    missing_by_station <= MAX_STATION_MISSING_FRACTION
].index

before_stations = df["station_id"].nunique()
df = df[df["station_id"].isin(keep_stations)].copy()

print(f"Stations: kept {df['station_id'].nunique()} of {before_stations}")
print("Rows remaining:", len(df))


In [ ]:
## Build a complete daily grid per station

# The lag features depend on consecutive calendar days, so every station needs one
# row per day across its own active span. We reindex each station from its first to
# its last date at daily frequency. Any day that was missing becomes an explicit NaN
# row, which is exactly what the next gap step works on. Station metadata (name, lat,
# lon, elevation) is constant per station so it is reattached after all stations are
# stacked back together.
df = df.sort_values(["station_id", "date"])

meta_cols = ["station_name", "island", "lat", "lon", "elevation_m"]
station_meta = df.groupby("station_id")[meta_cols].first()

frames = []
for station_id, group in df.groupby("station_id"):
    full_index = pd.date_range(group["date"].min(), group["date"].max(), freq="D")
    g = group.set_index("date")[["rainfall_mm"]].reindex(full_index)
    g.index.name = "date"
    g["station_id"] = station_id
    frames.append(g.reset_index())

daily = pd.concat(frames, ignore_index=True)

print("Daily grid rows:", len(daily))
print("Gaps to evaluate:", int(daily["rainfall_mm"].isna().sum()))


In [ ]:
## Interpolate short gaps, drop long gaps

# For each station, linearly interpolate runs of 1 to 2 missing days. limit caps the
# fill to MAX_GAP_DAYS, and limit_area="inside" means we only fill between real
# readings and never extend past the ends of a station record. Any NaN left after
# this belongs to a longer gap (or a record edge) and is dropped, since inventing
# several days of rainfall would pollute the lag features.
daily = daily.sort_values(["station_id", "date"])

daily["rainfall_mm"] = daily.groupby("station_id")["rainfall_mm"].transform(
    lambda s: s.interpolate(method="linear", limit=MAX_GAP_DAYS, limit_area="inside")
)

before = len(daily)
remaining_gaps = int(daily["rainfall_mm"].isna().sum())
daily = daily.dropna(subset=["rainfall_mm"]).reset_index(drop=True)

print("Long-gap rows dropped:", remaining_gaps)
print(f"Rows: kept {len(daily)} of {before}")


In [ ]:
## Reattach station metadata and finalize

# The gap work only carried station_id, date, and rainfall_mm. Here we join the
# constant station metadata back on, format the date as plain YYYY-MM-DD text, and
# order the columns to match the combined raw layout so downstream code stays simple.
clean = daily.merge(station_meta, on="station_id", how="left")
clean["date"] = clean["date"].dt.strftime("%Y-%m-%d")

clean = clean[
    [
        "date",
        "station_id",
        "station_name",
        "island",
        "lat",
        "lon",
        "elevation_m",
        "rainfall_mm",
    ]
]
clean = clean.sort_values(["station_id", "date"]).reset_index(drop=True)

print("Final shape:", clean.shape)
clean.head()


In [ ]:
## Export the cleaned dataset

# Write the cleaned daily rainfall table to the clean_data folder. This file feeds
# the rainfall lag feature engineering and the join to the cleaned water quality data.
clean.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote {len(clean):,} rows to {OUTPUT_PATH}")
print("Stations:", clean["station_id"].nunique())
print("Date range:", clean["date"].min(), "to", clean["date"].max())
